# Preprocessing

In [70]:
%load_ext autoreload
%autoreload 2
import os
from pathlib import Path
import numpy as np
import polars as pl
import pandas as pd
from mittens import GloVe
from utils import pipeline_v2, funcs
from transformers import pipeline
from sklearn.feature_extraction import DictVectorizer
from sklearn.model_selection import train_test_split
from collections import defaultdict

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
data_dir = Path("data")
data_dir.mkdir(parents=True, exist_ok=True)

subfolders = ["data-GloVe", "data-transformers", "data-RF", "data-w2v"]

for sub in subfolders:
    path = Path(data_dir) / sub
    path.mkdir(parents=True, exist_ok=True)

glove_dir=data_dir / "data-GloVe"
tran_dir=data_dir / "data-transformers"
rf_dir=data_dir / "data-RF"
w2v_dir=data_dir / "data-w2v"

## Reading and Splitting the Training Data

To optimize data preprocessing in the pipeline, we load the raw data into a **Polars DataFrame**, leveraging its multi-threaded engine to handle large files faster and more efficiently than pandas. 

To manage local memory constraints, the dataset is split into two equal partitions (`sample_df1` and `sample_df2`) and processed sequentially.

In [79]:
df_train = pl.read_csv(data_dir / 'train.csv')
df_test = pl.read_csv(data_dir / 'test.csv')

In [13]:
sample_df1, sample_df2 = train_test_split(
    df_train,
    train_size=0.5,
    stratify=df['tag'],
    random_state=80085
)

### Lyrics Preprocessing Overview

Each partition is mapped through `main_pipeline` via Polars' `map_elements` to clean and standardize the `lyrics` column:

- **Text Cleaning & Normalization:** Strips song metadata tags (e.g., *[Chorus]*, *[Verse]*), URLs, brackets, and punctuation, while expanding contractions.
- **Filtering & Tokenization:** Tokenizes text into individual words, discards non-alphabetic tokens, and removes standard English stopwords as well as common musical filler vocalizations (e.g., *"oh"*, *"yeah"*, *"la"*).
- **Linguistic Standardization:** Converts diacritics to ASCII, applies lowercasing, and lemmatizes tokens across multiple parts of speech.
- **Output Schema:** With `tokenized_output=True`, each song's lyrics are transformed into a list of cleaned tokens (`pl.List(pl.String)`) stored under `cleaned_lyrics`.

Once both batches finish processing, they are combined back into a single unified dataset (`cleaned_df`) using vertical concatenation (`pl.concat`).

In [ ]:
from utils.pipeline_v2 import main_pipeline

sample_df1 = sample_df1.with_columns(
    pl.col("lyrics")
    .map_elements(
        lambda x: main_pipeline(x, tokenized_output=True), #set to False for transformer
        return_dtype=pl.List(pl.String)#if tokenized_output=False this line should be removed
    )
    .alias("cleaned_lyrics")
)

sample_df2 = sample_df2.with_columns(
    pl.col("lyrics")
    .map_elements(
        lambda x: main_pipeline(x, tokenized_output=True), #set to False for transformer
        return_dtype=pl.List(pl.String) #if tokenized_output=False this line should be removed
    )
    .alias("cleaned_lyrics")
)

cleaned_df_train = pl.concat([sample_df1, sample_df2])

cleaned_df_test = df_test.with_columns(
    pl.col("lyrics")
    .map_elements(
        lambda x: main_pipeline(x, tokenized_output=True), #set to False for transformer
        return_dtype=pl.List(pl.String)#if tokenized_output=False this line should be removed
    )
    .alias("cleaned_lyrics"))

## Checkpointing & Class Distribution Overview

To preserve our preprocessed data and avoid redundant recomputation, we export the DataFrame to **Parquet format** and reload it. Parquet provides efficient column-oriented compression and natively preserves complex nested schemas, such as columns containing lists of strings (`pl.List(pl.String)`).

An initial inspection of the target genre distribution reveals a notable class imbalance, with **Rock**, **Pop**, and **Rap** making up the substantial majority of the dataset. Downstream modeling and evaluation strategies will need to account for this skew.

In [ ]:
cleaned_df_train.select(['tag', 'cleaned_lyrics']).write_parquet(tran_dir / 'cleaned_train_df.parquet')
cleaned_df_test.select(['cleaned_lyrics']).write_parquet(tran_dir / 'cleaned_test_df.parquet')

In [ ]:
cleaned_df_train.select(['tag', 'cleaned_lyrics']).write_parquet(w2v_dir / 'cleaned_train_df_tokenized.parquet')
cleaned_df_test.select(['cleaned_lyrics']).write_parquet(w2v_dir / 'cleaned_test_df_tokenized.parquet')

In [ ]:
cleaned_df_new = pl.read_parquet(w2v_dir / 'cleaned_train_df_tokenized.parquet')

In [20]:
cleaned_df_new['tag'].value_counts()

tag,count
str,u32
"""rb""",6203
"""rap""",38584
"""country""",3466
"""rock""",25332
"""misc""",5640
"""pop""",55742


In [ ]:
X = cleaned_df_new.select('cleaned_lyrics')
y = cleaned_df_new.select('tag')

In [68]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=80085, stratify=y
)

## Class Balancing via Controlled Downsampling

To resolve the genre distribution skew observed during EDA, we apply `balance_classes` to the training set. Rather than collapsing every category strictly to the minority class count, the routine samples each genre into a bounded range around the target threshold to retain class diversity. The balanced records are then split back into distinct `X` and `y` structures, before being passed through `the test_train_split` dropping the temporary helper column.
4. **Saving the Dataset** For posterity the dataset is saved to Parquet.

In [36]:
from utils.funcs import balance_classes

balanced_df = balance_classes(cleaned_df, class_col="tag", seed=80085)

In [ ]:
balanced_df.write_parquet(data_dir / 'balanced_train_df.parquet')

## Dataset Branching: Co-Occurrence Matrix Generation

Rather than restricting downstream modeling to a single architecture, we branch our data preparation to support varied embedding strategies, ranging from traditional distributional methods (Word2Vec, GloVe) to deep sequence models (Transformers).

### Co-Occurrence Computation
Count-based embeddings require pairwise co-occurrence frequencies across the corpus. Computing a full co-occurrence matrix naively can trigger SciPy out-of-memory errors due to quadratic vocabulary growth. 

We run `cooccurrence_matrix_generator` to slice matrix operations into manageable chunks, prune self-co-occurrences, and persist the output as a compact Parquet file (`data/cooccurrence_matrix.parquet`).

In [ ]:
from utils.funcs import cooccurrence_matrix_generator

df_cooccurrence = cooccurrence_matrix_generator(X_train, "cleaned_lyrics")
df_cooccurrence.write_parquet(glove_dir / "cooccurrence_matrix.parquet")

Computing pairs in blocks: 100%|██████████| 31/31 [00:38<00:00,  1.23s/it]


Assembling final co-occurrence table...


In [ ]:
cooc_matrix = pl.scan_parquet(glove_dir / 'cooccurrence_matrix.parquet')
# how many distinct words? (cheap, works on the integer codes)
print(cooc_matrix.select(pl.col("word_1").n_unique()).collect(engine="streaming"))

# the 100k most frequent words
top = (cooc_matrix.group_by("word_1").agg(pl.col("count").sum())
         .sort("count", descending=True).head(100_000)
         .collect(engine="streaming")["word_1"].cast(pl.String))
vocab = top.sort()
enum = pl.Enum(vocab)

# filter, encode to ids, and write to disk, all in chunks
(cooc_matrix.filter(pl.col("count") > 1)                          # drops most of the noise first
   .with_columns(pl.col("word_1", "word_2").cast(pl.String))
   .filter(pl.col("word_1").is_in(vocab) & pl.col("word_2").is_in(vocab))
   .select(pl.col("word_1").cast(enum).to_physical().cast(pl.Int32).alias("i"),
           pl.col("word_2").cast(enum).to_physical().cast(pl.Int32).alias("j"),
           pl.col("count").cast(pl.Float32).alias("x"))
   .sink_parquet(glove_dir / "pairs.parquet"))

vocab.to_frame("word").write_parquet(glove_dir / "vocab.parquet")
print("pairs kept:", pl.scan_parquet(glove_dir / "pairs.parquet").select(pl.len()).collect())

shape: (1, 1)
┌────────┐
│ word_1 │
│ ---    │
│ u32    │
╞════════╡
│ 150896 │
└────────┘


C:\Users\Lucas\AppData\Local\Temp\ipykernel_24356\795872482.py:19: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .sink_parquet("data/data-GloVe/pairs.parquet"))


pairs kept: shape: (1, 1)
┌──────────┐
│ len      │
│ ---      │
│ u32      │
╞══════════╡
│ 91044552 │
└──────────┘


In [ ]:
pairs = pl.read_parquet(glove_dir / "pairs.parquet")
i, j, x = pairs["i"].to_numpy(), pairs["j"].to_numpy(), pairs["x"].to_numpy()
del pairs

idx = np.arange(len(x), dtype=np.int32)
np.random.default_rng(0).shuffle(idx)
np.save(glove_dir / "i.npy", i[idx]); np.save(glove_dir / "j.npy", j[idx]); np.save(glove_dir / "x.npy", x[idx])
del i, j, x, idx

In [ ]:
vocab = pl.read_parquet("vocab.parquet")["word"]
from utils.GloVe import GloVe
glove = GloVe(vocab, dim=25).fit(glove_dir / "i.npy", glove_dir / "j.npy", glove_dir / "x.npy", epochs=20)
print(glove.most_similar("love"))
glove.save(glove_dir / "glove_25d")

c:\Users\Lucas\Github\Music-Genre-Identifier\utils\GloVe.py:73: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:209.)
  return torch.as_tensor(np.asarray(arr), dtype=dtype, device=self.device)


epoch 1/20  loss 0.1577
epoch 2/20  loss 0.0213
epoch 3/20  loss 0.0194
epoch 4/20  loss 0.0164
epoch 5/20  loss 0.0143
epoch 6/20  loss 0.0134
epoch 7/20  loss 0.0128
epoch 8/20  loss 0.0123
epoch 9/20  loss 0.0119
epoch 10/20  loss 0.0116
epoch 11/20  loss 0.0114
epoch 12/20  loss 0.0113
epoch 13/20  loss 0.0111
epoch 14/20  loss 0.0109
epoch 15/20  loss 0.0108
epoch 16/20  loss 0.0107
epoch 17/20  loss 0.0105
epoch 18/20  loss 0.0104
epoch 19/20  loss 0.0103
epoch 20/20  loss 0.0102
[('feel', 0.994), ('let', 0.994), ('leave', 0.993), ('time', 0.993), ('way', 0.992), ('go', 0.992), ('come', 0.992), ('never', 0.992)]


In [59]:
vectors, vocab = GloVe.load_vectors("data/data-GloVe/glove_25d")
index = {word: k for k, word in enumerate(vocab)}

In [60]:
def song_vector(tokens, vectors, index, idf=None):
    ids = [index[t] for t in tokens if t in index]
    if not ids:
        return np.zeros(vectors.shape[1], dtype=np.float32)
    if idf is None:
        return vectors[ids].mean(axis=0)
    weights = np.array([idf[t] for t in tokens if t in index], dtype=np.float32)
    return np.average(vectors[ids], axis=0, weights=weights)

In [69]:
X_train_glove = np.vstack([song_vector(t, vectors, index) for t in X_train['cleaned_lyrics']])
X_val_glove = np.vstack([song_vector(t, vectors, index) for t in X_val['cleaned_lyrics']])

print(X_train_glove.shape)

(101225, 25)
